# Movement Smoothness Analysis — Data Processing Pipeline

**Project:** Study of movement smoothness in post-stroke patients.

**Author:** Pichenot Nolan

**Laboratory:** EuroMov Digital Health in Motion

---

## Overview

**Role:** This notebook constitutes the central data processing backbone, ingesting raw kinematic recordings (`.dat`) and outputting standardized tabular exports for statistical analysis.

**Method:** It parses participant metadata, discards pre-task baselines, automatically segments reach-to-target movements, derives core kinematic signals (velocity, acceleration), and extracts validated smoothness metrics (SPARC, submovements).

**Rationale:** Standardizing feature extraction in Python guarantees data integrity and modularity before downstream statistical analysis and visualization in R (`02_analyse_statistique.Rmd`).

### Pipeline Exports

| Export File | Granularity | Contents |
|---|---|---|
| `metrics_smoothness_export.csv` | One row per segment | SPARC, peak/mean velocity, duration, submovement count, point count |
| `trajectories_export.csv` | One row per time point | Position (x, y), time, velocity, acceleration, target geometry, inside target flag |

### Pipeline Overview Architecture

```text
Raw .dat files
      │
      ▼
[1] Setup                            → Imports & group label configuration
[2] Metadata Extraction              → Parse subject ID, cohort group & trial ID
[3] Data Loading & Preprocessing     → Baseline truncation via target coordinate shifts
[4] Movement Segmentation            → Trial splitting at target changes
[5] Kinematic Signal Derivation      → Euclidean velocity & acceleration profiles
[6] Submovement Detection            → Corrective peak counting (5% threshold)
[7] Smoothness Metric — SPARC        → Spectral Arc Length computation
[8] Batch Processing Pipeline        → Directory setup & main dataset aggregation loop
[9] Data Export                      → Generation of metrics & trajectories CSV files
```


## 1. Setup — Imports and Configuration

**Role:** Initializes the computational environment and defines global constants.

**Method:** Imports file system navigation tools (`os`, `pathlib`), numerical processing libraries (`numpy`, `pandas`), and establishes `GROUP_MAP` for cohort labeling.

**Rationale:** Standardizing group labels at the top level prevents hard-coding errors downstream and ensures the vocabulary used in exported datasets precisely matches the project's clinical definitions.

In [ ]:
import os
import csv
import numpy as np
import pandas as pd
from pathlib import Path

# Standardizes naming conventions to match clinical statistical grouping
GROUP_MAP = {"ap": "Aged_Control", "cp": "Young_Control", "pp": "Stroke_Paretic"}

## 2. Metadata Extraction

**Role:** Identifies the subject identifier, cohort group, and trial number directly from the raw file name.

**Method:** Parses strings formatted as `xx##-#.dat` using string slicing to extract the two-letter group code, subject number, and trial ID, mapping the code against `GROUP_MAP`.

**Rationale:** Embedding metadata within the filename eliminates the dependency on external registries or manual data entry, reducing the risk of cohort mismatch errors during batch processing.

In [ ]:
def extract_metadata(filename):
    """
    Extract participant metadata from raw data filenames.

    Parameters
    ----------
    filename : str
        The base filename to be parsed.

    Returns
    -------
    tuple of (str, str, str) or None
        Returns (subject_id, group_label, trial_number), or None if invalid.
    """
    file_stem = os.path.splitext(filename)[0]
    group_code = file_stem[:2].lower()
    
    # Validate format strictly to prevent silent failures during large batch processing
    if group_code not in GROUP_MAP:
        print(f"Warning: '{filename}' does not contain a recognized group code.")
        return None
    if "-" not in file_stem:
        print(f"Warning: '{filename}' lacks the required subject-trial delimiter ('-').")
        return None
    
    subject_id, trial_number = file_stem[2:].split("-", 1)
    
    if not subject_id.isdigit() or not trial_number.isdigit():
        print(f"Warning: Non-numeric identifier found in '{filename}'.")
        return None

    return subject_id, GROUP_MAP[group_code], trial_number

## 3. Data Loading and Preprocessing

**Role:** Isolates the active motor execution phase from the continuous raw recording.

**Method:** Reads the raw `.dat` file into a NumPy array and truncates all spatial observations prior to the first detection of a target coordinate shift.

**Rationale:** Participants naturally display static resting states before a trial physically begins. Discarding this pre-task baseline ensures that kinematic metrics reflect purely intentional movement rather than hardware noise or resting tremor.

In [ ]:
def locate_target_shifts(data_array):
    """
    Identify sample indices where the target position coordinates change.

    Parameters
    ----------
    data_array : numpy.ndarray
        Raw kinematic array containing [x, y, t_ms, target_x, target_y, ...].

    Returns
    -------
    numpy.ndarray
        Array of index locations corresponding to target coordinate changes.
    """
    target_x = data_array[:, 3]
    target_y = data_array[:, 4]
    return np.where((np.diff(target_x) != 0) | (np.diff(target_y) != 0))[0] + 1

def load_and_truncate(file_path):
    """
    Load a raw recording and remove the pre-task baseline period.

    Parameters
    ----------
    file_path : str or pathlib.Path
        Path to the recording file.

    Returns
    -------
    numpy.ndarray or None
        Truncated 2D data array starting from task onset, or None if invalid.
    """
    # Rely on pandas for robust parsing of heavily commented header structures
    df = pd.read_csv(file_path, sep=r"\s+", engine="python", header=0, comment="#")
    data_array = df.values

    shift_indices = locate_target_shifts(data_array)
    if len(shift_indices) == 0:
        print(f"Warning: No target shift detected in '{file_path}'.")
        return None
    
    # Truncate to discard uninformative pre-task rest periods
    return data_array[shift_indices[0]:, :]

## 4. Movement Segmentation

**Role:** Segments the continuous truncated recording into discrete, independent reach-to-target trials.

**Method:** Splits the main temporal array at every index where the target location changes, generating a list of distinct sub-arrays.

**Rationale:** Smoothness metrics like SPARC are only valid when applied to single, goal-directed movements. Segmentation is strictly required to prevent merging distinct motor plans.

In [ ]:
def split_into_segments(data_array):
    """
    Split a truncated recording into discrete reach-to-target segments.

    Parameters
    ----------
    data_array : numpy.ndarray
        Truncated kinematic array.

    Returns
    -------
    list of numpy.ndarray
        List containing separate 2D arrays for each individual trial segment.
    """
    shift_indices = locate_target_shifts(data_array)
    return np.split(data_array, shift_indices)

## 5. Velocity Profile Computation

**Role:** Derives Euclidean velocity from the raw x/y position coordinates across time.

**Method:** Employs `numpy.gradient` (central difference algorithm) to differentiate spatial coordinates with respect to the actual temporal timestamps.

**Rationale:** Velocity is the foundational kinematic signal for smoothness analysis. Utilizing temporal gradients rather than fixed increments (`dt`) prevents compounding errors caused by minor fluctuations in the hardware sampling rate.

In [ ]:
def compute_velocity_profile(segment_data):
    """
    Derive Euclidean speed and normalized time arrays for a given segment.

    Parameters
    ----------
    segment_data : numpy.ndarray
        2D array for a single segment.

    Returns
    -------
    tuple of numpy.ndarray
        Returns (x_pos, y_pos, time_seconds, velocity_profile).
    """
    x_pos = segment_data[:, 0]
    y_pos = segment_data[:, 1]

    # Zero-anchor time to ensure metrics evaluate pure movement duration
    time_s = (segment_data[:, 2] - segment_data[0, 2]) / 1000.0

    # Central differences automatically account for non-uniform sampling intervals
    vel_x = np.gradient(x_pos, time_s)
    vel_y = np.gradient(y_pos, time_s)
    velocity_profile = np.sqrt(vel_x**2 + vel_y**2)
    
    return x_pos, y_pos, time_s, velocity_profile

## 6. Submovement Detection

**Role:** Quantifies the number of corrective motor commands (submovements) executed during a reach.

**Method:** Identifies local velocity maxima bounded by local minima, filtering out peaks that fall below a defined baseline noise threshold (5% of peak velocity).

**Rationale:** Submovement count provides a tangible, time-domain metric of motor fragmentation. Neurologically impaired patients typically rely on sequential corrective submovements, making this a direct proxy for clinical motor deficit.

In [ ]:
def count_submovements(velocity_profile, threshold_ratio=0.05):
    """
    Count the number of corrective submovements in a velocity profile.

    Parameters
    ----------
    velocity_profile : numpy.ndarray
        Euclidean speed signal.
    threshold_ratio : float
        Fraction of peak velocity defining the activity threshold (default 0.05).

    Returns
    -------
    int
        Number of valid submovements detected.
    """
    if velocity_profile is None or len(velocity_profile) < 3:
        return 0

    # Establish a dynamic threshold to ignore baseline hardware noise
    activity_threshold = threshold_ratio * np.max(velocity_profile)

    peaks, valleys = [], []
    for i in range(1, len(velocity_profile) - 1):
        if velocity_profile[i] >= velocity_profile[i - 1] and velocity_profile[i] > velocity_profile[i + 1]:
            peaks.append(i)
        elif velocity_profile[i] <= velocity_profile[i - 1] and velocity_profile[i] < velocity_profile[i + 1]:
            valleys.append(i)
            
    peaks, valleys = np.array(peaks, dtype=int), np.array(valleys, dtype=int)

    # Restrict evaluation to the active movement phase
    active_indices = np.where(velocity_profile >= activity_threshold)[0]
    if len(active_indices) == 0:
        return 0

    peaks = peaks[(peaks >= active_indices[0]) & (peaks <= active_indices[-1])]
    valleys = valleys[(valleys >= active_indices[0]) & (valleys <= active_indices[-1])]

    submovement_count = 0
    current_evaluation_idx = int(active_indices[0])
    
    while True:
        # Identify the next valid peak representing a potential submovement
        subsequent_peaks = peaks[peaks > current_evaluation_idx]
        if len(subsequent_peaks) == 0:
            break
        peak_idx = int(subsequent_peaks[0])
        submovement_count += 1
        
        # Locate the terminating local minimum of this specific submovement
        subsequent_valleys = valleys[valleys > peak_idx]
        if len(subsequent_valleys) == 0:
            break
        
        # Advance the search window to avoid duplicate counting
        current_evaluation_idx = int(subsequent_valleys[0])
        if current_evaluation_idx >= len(velocity_profile) - 1:
            break

    # A completed movement inherently implies at least one baseline submovement
    return max(submovement_count, 1)

## 7. Smoothness Metric: Spectral Arc Length (SPARC)

**Role:** Quantifies overall movement smoothness using frequency-domain characteristics.

**Method:** Normalizes the velocity, performs a zero-padded Fast Fourier Transform (FFT), and integrates the resulting spectral arc length up to a physiological limit (10 Hz).

**Rationale:** Time-domain metrics (like jerk) are highly sensitive to sampling noise. SPARC provides a robust, dimensionless assessment (closer to 0 is smoother) that scales effectively across distinct populations (stroke vs. healthy), serving as the primary biomarker for this study.

In [ ]:
def compute_sparc_score(velocity_profile, fs, amplitude_threshold=0.05, pad_level=4):
    """
    Compute the Spectral Arc Length (SPARC) score from a kinematic velocity profile.
    
    Parameters
    ----------
    velocity_profile : numpy.ndarray
        The Euclidean speed signal.
    fs : float
        The sampling frequency (Hz).
    amplitude_threshold : float
        Threshold for spectrum truncation (default 0.05).
    pad_level : int
        Zero-padding factor for FFT resolution enhancement (default 4).
    
    Returns
    -------
    float
        SPARC score. Negative value where proximity to 0 indicates higher smoothness.
    """
    # Normalization guarantees a dimensionless metric invariant to movement amplitude
    normalized_velocity = velocity_profile / np.max(velocity_profile)
    dt = 1.0 / fs

    # Zero-padding increases frequency domain resolution, stabilizing the integral
    base_power = np.ceil(np.log2(len(normalized_velocity)))
    fft_length = int(2**(base_power + pad_level))

    frequencies = np.fft.rfftfreq(n=fft_length, d=dt)
    amplitude_spectrum = np.abs(np.fft.rfft(normalized_velocity, n=fft_length))
    amplitude_spectrum = amplitude_spectrum / amplitude_spectrum[0]

    # Truncate the spectrum to isolate physiological movement from high-frequency noise
    valid_indices = np.where(amplitude_spectrum > amplitude_threshold)[0]
    if len(valid_indices) == 0:
        return 0.0
        
    cutoff_index = max(valid_indices)
    trimmed_frequencies = frequencies[:cutoff_index]
    trimmed_amplitude = amplitude_spectrum[:cutoff_index]
    cutoff_frequency = frequencies[cutoff_index]

    # Geometrically calculate the arc length of the spectral curve
    amplitude_gradient = np.diff(trimmed_amplitude) / np.diff(trimmed_frequencies)
    arc_length_integrand = np.sqrt((1.0 / cutoff_frequency)**2 + amplitude_gradient**2)
    sparc_score = -np.trapezoid(arc_length_integrand, trimmed_frequencies[:-1])

    return float(sparc_score)

## 8. Batch Processing Pipeline
### 8.1 Path Configuration

**Role:** Establishes input and output directories dynamically.

**Method:** Uses `pathlib` to construct folder trees relative to the script's execution location.

**Rationale:** Prevents "file not found" errors when transitioning the project between environments or operating systems.

In [ ]:
BASE_DIR = Path.cwd()
DATA_FOLDER = BASE_DIR / "data"
EXPORTS_FOLDER = BASE_DIR / "results"

# Guarantee destination existence to avoid script termination during export
EXPORTS_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_METRICS = EXPORTS_FOLDER / "metrics_smoothness_export.csv"
OUTPUT_CSV_TRAJECTORIES = EXPORTS_FOLDER / "trajectories_export.csv"

### 8.2 Main Execution Loop

**Role:** Orchestrates the entire pipeline over the full dataset to generate the finalized data structures.

**Method:** Iterates through every `.dat` file, applies the preprocessing/segmentation functions, computes localized metrics for each segment, and aggregates the results into comprehensive dictionaries.

**Rationale:** Centralized batch execution transforms unstructured, fragmented raw data into two synchronized datasets (trial-level metrics and point-level trajectories), feeding directly into statistical analysis.

In [ ]:
segment_metrics = []
trajectory_metrics = []

for file_path in sorted(DATA_FOLDER.glob("*.dat")):
    metadata = extract_metadata(file_path.name)
    if metadata is None:
        continue
    subject_id, group_label, trial_number = metadata

    raw_data = load_and_truncate(file_path)
    if raw_data is None:
        continue

    segments = split_into_segments(raw_data)

    for segment_idx, segment_data in enumerate(segments, start=1):
        # Filter invalid fragments caused by momentary recording glitches
        if len(segment_data) < 5:
            continue

        x_pos, y_pos, time_s, velocity_profile = compute_velocity_profile(segment_data)
        
        dt = np.median(np.diff(time_s))
        fs = 1.0 / dt if dt > 0 else 33.33  # Default based on hardware specs if dt fails

        accel_x = np.gradient(np.gradient(x_pos, time_s), time_s)
        accel_y = np.gradient(np.gradient(y_pos, time_s), time_s)
        acceleration_profile = np.sqrt(accel_x**2 + accel_y**2)

        n_submovements = count_submovements(velocity_profile)
        sparc_score = compute_sparc_score(velocity_profile, fs)

        # Aggregate trial-level parameters for group comparisons
        segment_metrics.append({
            "Subject_ID": subject_id,
            "Group": group_label,
            "Trial_Num": trial_number,
            "Segment_Num": segment_idx,
            "N_Points": len(segment_data),
            "Duration_s": round(float(time_s[-1] - time_s[0]), 3),
            "Peak_Velocity": float(np.max(velocity_profile)),
            "Mean_Velocity": float(np.mean(velocity_profile)),
            "N_Submovements": n_submovements,
            "SPARC": sparc_score,
        })

        target_x = float(segment_data[0, 3])
        target_y = float(segment_data[0, 4])
        target_r = float(segment_data[0, 5])

        # Compile granular coordinates for trajectory visualizations
        for i in range(len(x_pos)):
            trajectory_metrics.append({
                "Subject_ID": subject_id,
                "Group": group_label,
                "Trial_Num": trial_number,
                "Segment_Num": segment_idx,
                "point_index": i + 1,
                "x": float(x_pos[i]),
                "y": float(y_pos[i]),
                "t_s": round(float(time_s[i]), 6),
                "velocity": float(velocity_profile[i]),
                "acceleration": float(acceleration_profile[i]),
                "target_x": target_x,
                "target_y": target_y,
                "target_r": target_r,
                "inside_target": int(segment_data[i, 6])
            })

    print(f"Processed {file_path.name}: {len(segments)} valid segments.")

## 9. Data Export

**Role:** Writes the compiled datasets to permanent storage.

**Method:** Uses `csv.DictWriter` to output the accumulated Python dictionaries into structured `.csv` files.

**Rationale:** Preserves the results in a tidy, universal format, enabling seamless integration with the R ecosystem for visualization and statistical analysis.

In [ ]:
if segment_metrics:
    with open(OUTPUT_CSV_METRICS, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=segment_metrics[0].keys())
        writer.writeheader()
        writer.writerows(segment_metrics)
    print(f"Export complete: {len(segment_metrics)} rows -> {OUTPUT_CSV_METRICS.name}")
else:
    print("Warning: No metric data available for export.")

if trajectory_metrics:
    with open(OUTPUT_CSV_TRAJECTORIES, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=trajectory_metrics[0].keys())
        writer.writeheader()
        writer.writerows(trajectory_metrics)
    print(f"Export complete: {len(trajectory_metrics)} rows -> {OUTPUT_CSV_TRAJECTORIES.name}")
else:
    print("Warning: No trajectory data available for export.")